# 🛰️ Punjab GeoGuardian - AI-Based Monitoring of Urban Expansion and Farmland Loss
**AIRI Team PITB - AI Internship Task 3** · End-to-End GeoAI and Satellite Intelligence Project
**Prepared by:** Hamza Nadeem · **Submitted to:** Omar Farooq - Associate AI/ML Engineer

This notebook runs the complete GeoAI workflow with **Google Earth Engine + Sentinel-2** and Python:

`Select AOI → Load Sentinel-2 → Cloud-free composite → NDVI & NDBI → Classify land cover → Compare 2019 vs 2025 → Detect changed pixels → Calculate area → Maps & dashboard → Validation & error analysis → Report`

| Notebook part | PDF requirement |
|---|---|
| 1-2 Setup & AOI | §5 AOI selection, §11 data collection |
| 3 Data collection | §11-12 Sentinel-2, date/cloud filter, median composite, clip |
| 4 Pre-processing maps | §12 RGB / NDVI / NDBI for both years (Week-1 deliverables) |
| 5 Training samples | §14 ≥ 50 samples per class, spread over the AOI |
| 6 Classification | §13 Option A (rules) + Option B (Random Forest / XGBoost) |
| 7 Change detection | §15 agri-to-urban conversion, area table, hotspots |
| 8 Validation | §18 ≥ 50 manually checked points, accuracy |
| 9 Error analysis | §19 ≥ 15 cases + improvements |
| 10 Maps | §16 all required maps (≥ 8) |
| 11 Interactive map / dashboard | §17 Folium layer toggles + Streamlit |
| 12 Exports, report, LinkedIn | §21 deliverables, §25 portfolio section |

> ⚠️ Learning prototype - **not** an official government monitoring system. Results are *possible / estimated* changes.

**Runtime:** about 15-25 min for the first run (most of it is the Earth Engine download). Re-runs take about 2 min.

## 1. Setup

In [ ]:
# --- Settings you may change -------------------------------------------------------
EE_PROJECT   = "your-gcp-project-id"   # <-- your Google Cloud project registered for Earth Engine
USE_DRIVE    = True                    # keep data and outputs on Google Drive (= Drive submission)
PROJECT_DIR  = "/content/drive/MyDrive/punjab-geoguardian"   # where the project folder lives
REPO_URL     = ""                      # optional: your GitHub repo URL (used if PROJECT_DIR is missing)
CONFIG_FILE  = "config/lahore_raiwind.json"
USE_SYNTHETIC = False                  # True = offline smoke test only (watermarked, not real results)

In [ ]:
import os, sys, subprocess
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB and USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
if not os.path.exists(PROJECT_DIR):
    if REPO_URL:
        subprocess.run(["git", "clone", REPO_URL, PROJECT_DIR], check=True)
    else:
        raise FileNotFoundError(f"Upload the project folder to {PROJECT_DIR} (or set REPO_URL).")
os.chdir(PROJECT_DIR)
sys.path.insert(0, PROJECT_DIR)
if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
print("Working directory:", os.getcwd())

## 2. Area of Interest and configuration

In [ ]:
import json, pandas as pd
from IPython.display import Image, display, Markdown, HTML
from geoguardian.config import ProjectConfig
from geoguardian.pipeline import GeoGuardian

if USE_SYNTHETIC:
    from geoguardian.synthetic import make_synthetic_stack
    cfg = ProjectConfig(aoi_key="synthetic_demo", aoi_name="Synthetic test scene", synthetic=True,
                        data_dir="data_synthetic", output_dir="outputs_synthetic")
else:
    cfg = ProjectConfig.from_json(CONFIG_FILE)
cfg.validate()
pd.DataFrame({
    "Setting": ["Author", "AOI", "AOI mode", "Centre (lat, lon)", "Size", "Years compared",
                "Extra years (trend)", "Season (MM-DD)", "Cloud filter", "Satellite"],
    "Value": [cfg.author_name, cfg.aoi_name, cfg.aoi_mode, f"{cfg.center_lat}, {cfg.center_lon}",
              f"{cfg.width_km} x {cfg.height_km} km", f"{cfg.baseline_year} and {cfg.recent_year}",
              cfg.extra_years, f"{cfg.season_start} to {cfg.season_end}",
              f"< {cfg.max_cloud_pct}% scene cloud + Cloud Score+ pixel mask",
              "Sentinel-2 L2A harmonized (10 m)"]})

**Why this AOI?** The Raiwind Road corridor in south-western Lahore is a well-known peri-urban growth belt (the PDF's own example). Farmland, brick kilns and new housing schemes sit side by side there. To analyse another city or a full district, change `aoi_key`, `center_lat/lon` or `aoi_mode="district"` + `district_name` in the config file (presets: `geoguardian.config.AOI_PRESETS`).

In [ ]:
if not USE_SYNTHETIC:
    from geoguardian.gee_data import init_earth_engine, preview_map
    init_earth_engine(EE_PROJECT)
    m = preview_map(cfg)   # live Earth Engine layers with toggles + Google satellite basemap
    display(m)             # 📸 Week-1 deliverable: screenshot the AOI boundary from this map

## 3. Data collection (Sentinel-2 via Google Earth Engine)
Cloud-masked **median composites** for each year (same Feb-Mar season), clipped to the AOI and downloaded as one GeoTIFF stack on a 10 m UTM grid.

In [ ]:
FORCE_DOWNLOAD = False
if USE_SYNTHETIC:
    make_synthetic_stack(cfg)
elif FORCE_DOWNLOAD or not cfg.stack_path.exists():
    from geoguardian.gee_data import collect_data
    collect_data(cfg)
meta = json.loads(cfg.stack_meta_path.read_text())
pd.DataFrame([{ "Year": y, "Date range": f"{i['start']} to {i['end']}", "Image dates": i["n_dates"],
                "Scenes": i["n_scenes"], "Tiles": ", ".join(sorted(set(i["tiles"]))),
                "Window widened (days)": i["window_extended_days"]}
              for y, i in meta["image_info"].items()])

## 4. Pre-processing outputs - RGB, NDVI, NDBI (Week-1 deliverables)

In [ ]:
project = GeoGuardian(cfg)
project.load().preprocessing_maps()
F = project.figures
for k in ["aoi", "rgb_pair", "ndvi_b", "ndvi_r", "ndbi_b", "ndbi_r"]:
    display(Image(filename=str(F[k]), width=900 if k in ("aoi", "rgb_pair") else 560))

**Reading the indices**
- **NDVI = (NIR − Red)/(NIR + Red)**: green crops and trees are high (≈0.4-0.9); built-up and bare soil are low (≈0.1-0.2); water is negative.
- **NDBI = (SWIR − NIR)/(SWIR + NIR)**: built-up surfaces are high, vegetation is low. Dry bare soil is *also* high, so NDBI alone confuses fallow fields with construction. That is why NDVI, NDBI, MNDWI and all bands are combined in a classifier.

## 5. Training samples (≥ 50 per class)

In [ ]:
project.build_training()
s = project.samples
display(s.groupby("class_name").agg(samples=("sample_id", "count"), spatial_blocks=("block", "nunique")))
display(s.groupby(["class_name", "source"]).size().rename("samples").to_frame())
print("Visual spot-check sheet:", project.p.validation / "training_samples_review.pdf",
      "(created in step 8). To drop a bad sample, set keep=False in",
      project.p.validation / "training_samples_review.csv", "and re-run.")

## 6. Land classification - Option A (rules) vs Option B (Random Forest / XGBoost)

In [ ]:
project.classify()
cv = pd.DataFrame({k: {m: v for m, v in d.items() if m != "confusion_matrix"}
                   for k, d in project.model.all_cv.items()}).T
display(cv)
print("Selected model:", project.model.name)
print("Tuned rule thresholds:", project.model.rule["tuned"])
display(project.model.importances.rename("importance").to_frame())

## 7. Change detection, area calculation and hotspots

In [ ]:
from geoguardian.change import hotspot_table
project.detect_change()
display(project.change.stats)
display(Markdown(f"**Transition matrix {cfg.baseline_year} → {cfg.recent_year} (km²)**"))
display(project.change.transition)
display(project.change.timeseries)
display(hotspot_table(project.change, project.scene.grid, 10)[
    ["rank", "converted_km2", "converted_pct_of_cell", "location", "center_lat", "center_lon"]])

## 8. Validation - 60 manually checked points (PDF minimum 50)
1. The next cell draws 60 stratified points (15 vegetation, 15 built-up, 15 other, 15 detected conversion), all ≥ 100 m from training samples.
2. **Label them by eye** with the widget below. For each point, look at the 2019/2025 chips and click the Google Maps link for high-resolution imagery. Choose the class you see in 2025 and whether vegetation became built-up. The model's prediction is hidden, so your labels are unbiased.
3. Labels are saved to `outputs/validation/validation_points.csv` (you can also edit it in Google Sheets). Then run the cells after the widget.

In [ ]:
project.validation_sheet()
from geoguardian.labeler import ValidationLabeler
ValidationLabeler(project.p.validation / "validation_points.csv", project.p.validation / "chips").show()

In [ ]:
# Run this after labelling (it reloads the CSV). Unlabelled points are marked PROVISIONAL.
project.assess()
v = project.validation
print(f"Points: {v['n_points']} | manual: {v['n_manual']} | correct: {v['correct']} | incorrect: {v['incorrect']}")
print(f"Approximate accuracy: {100*v['overall_accuracy']:.1f}%  (area-weighted {100*v['area_weighted_overall_accuracy']:.1f}%), kappa {v['kappa']:.3f}")
print(f"Rule-based (Option A) on the same points: {100*project.rule_validation['overall_accuracy']:.1f}%")
display(v["per_class"]); display(v["confusion_matrix"])
display(pd.DataFrame([v["change"]]).T.rename(columns={0: "value"}))
display(v["table"])

## 9. Error analysis (≥ 15 cases)

In [ ]:
e = project.errors
display(e["table"])
display(Markdown("**Most common reasons:** " + ", ".join(f"{k} ({n})" for k, n in e["reason_counts"].items())))
display(Markdown("**Suggested improvements:**\n" + "\n".join(f"- {i}" for i in e["improvements"])))

✏️ The reasons are *suggested* from pixel evidence. Check each case against its chip (`outputs/validation/chips/point_XXX.png`) and add your own comments in the `notes` column of the validation CSV. They appear in the final tables.

## 10. All maps

In [ ]:
project.make_maps()
F = project.figures
order = ["class_b", "class_r", "class_pair", "change", "change_classes", "hotspots", "dndvi", "dndbi",
         "heat", "rule_r", "samples", "val_pts", "areas", "trend", "transition", "models", "importance",
         "cv_cm", "val_cm"]
for k in order:
    if k in F:
        display(Image(filename=str(F[k]), width=620))

## 11. Interactive map (Folium, layer toggles) and Streamlit dashboard

In [ ]:
project.make_web_map()
display(HTML(project.web_map.read_text()))   # toggle layers in the top-right control

**Streamlit dashboard** - AOI name, year selection, RGB/NDVI/NDBI, change map, area table and observations.
Run the cell below and open the link it prints. To publish it, push the repo (including `outputs/`) to GitHub and deploy `app/streamlit_app.py` on [Streamlit Community Cloud](https://streamlit.io/cloud).

In [ ]:
import subprocess, time
env = dict(os.environ, GEOGUARDIAN_OUTPUTS=str(cfg.out.resolve()))
subprocess.Popen(["streamlit", "run", "app/streamlit_app.py", "--server.port", "8501",
                  "--server.headless", "true", "--server.enableCORS", "false",
                  "--server.enableXsrfProtection", "false"], env=env)
time.sleep(6)
if IN_COLAB:
    from google.colab.output import serve_kernel_port_as_window
    serve_kernel_port_as_window(8501)   # opens the dashboard in a new browser tab

## 12. Exports, Week-1 summary, final report PDF and LinkedIn section

In [ ]:
project.week1_summary().export().report()
display(Markdown(f"**Final report:** `{next(project.p.report.glob('*.pdf'))}`"))
display(Markdown((project.p.report / "LINKEDIN_PORTFOLIO.md").read_text()))

In [ ]:
# Zip everything for submission (Drive folder already contains all files if USE_DRIVE=True)
zip_path = project.zip_outputs()
print("Submission zip:", zip_path)

### ✅ Submission checklist
- [ ] All 60 validation points labelled manually (no *PROVISIONAL* banner in the report)
- [ ] Error-analysis notes reviewed
- [ ] Screenshots: AOI map (part 2), interactive map (part 11), dashboard (part 11)
- [ ] Notebook saved **with outputs** (File → Save) and shared
- [ ] Repo pushed to GitHub (see README) or the Drive folder shared with the supervisor
- [ ] LinkedIn post published from `outputs/report/LINKEDIN_PORTFOLIO.md`